In [1]:
!pip install pandas numpy scikit-learn tqdm

In [2]:
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score, log_loss

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
CSV_PATH = "../../dataset/questions.csv"   # change this if needed

df = pd.read_csv(CSV_PATH)
df = df[["question1", "question2", "is_duplicate"]].dropna().reset_index(drop=True)

print(df.shape)
df.head()

(404348, 3)


,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
USE_SUBSET = True
SUBSET_SIZE = 50000

if USE_SUBSET:
    df = df.sample(n=min(SUBSET_SIZE, len(df)), random_state=SEED).reset_index(drop=True)

print(df.shape)
df["is_duplicate"].value_counts()

(50000, 3)


is_duplicate
0    31810
1    18190
Name: count, dtype: int64

In [6]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "am", "was", "were", "be", "been", "being",
    "do", "does", "did", "doing", "have", "has", "had", "having",
    "i", "you", "he", "she", "it", "we", "they", "me", "him", "her", "them",
    "my", "your", "his", "their", "our",
    "and", "or", "but", "if", "then", "else", "for", "to", "of", "in", "on", "at", "by", "with", "as",
    "this", "that", "these", "those",
    "what", "which", "who", "whom", "why", "how", "when", "where",
    "can", "could", "should", "would", "will", "shall", "may", "might", "must",
    "not", "no", "nor", "so", "than", "too", "very"
}

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def unique_in_order(tokens):
    seen = set()
    out = []
    for tok in tokens:
        if tok not in seen:
            out.append(tok)
            seen.add(tok)
    return out

def extract_keywords(tokens):
    keywords = [tok for tok in tokens if tok not in STOPWORDS]
    return keywords if keywords else tokens[:]

In [7]:
tqdm.pandas()

df["q1_tokens"] = df["question1"].progress_apply(tokenize)
df["q2_tokens"] = df["question2"].progress_apply(tokenize)

df["q1_unique"] = df["q1_tokens"].progress_apply(unique_in_order)
df["q2_unique"] = df["q2_tokens"].progress_apply(unique_in_order)

df["q1_keywords"] = df["q1_tokens"].progress_apply(extract_keywords)
df["q2_keywords"] = df["q2_tokens"].progress_apply(extract_keywords)

df.head()

100%|██████████| 50000/50000 [00:00<00:00, 983811.68it/s]


,question1,question2,is_duplicate,q1_tokens,q2_tokens,q1_unique,q2_unique,q1_keywords,q2_keywords
0,Do people realize that you can send marijuana ...,How do you send weed through the mail?,0,"[do, people, realize, that, you, can, send, ma...","[how, do, you, send, weed, through, the, mail]","[do, people, realize, that, you, can, send, ma...","[how, do, you, send, weed, through, the, mail]","[people, realize, send, marijuana, through, ov...","[send, weed, through, mail]"
1,How can rock music be brought back?,What would it take for rock music to make a co...,1,"[how, can, rock, music, be, brought, back]","[what, would, it, take, for, rock, music, to, ...","[how, can, rock, music, be, brought, back]","[what, would, it, take, for, rock, music, to, ...","[rock, music, brought, back]","[take, rock, music, make, come, back]"
2,Why does one feel relaxed after smoking a join...,How do I sober up quickly after smoking weed/m...,0,"[why, does, one, feel, relaxed, after, smoking...","[how, do, i, sober, up, quickly, after, smokin...","[why, does, one, feel, relaxed, after, smoking...","[how, do, i, sober, up, quickly, after, smokin...","[one, feel, relaxed, after, smoking, joint, ma...","[sober, up, quickly, after, smoking, weed, mar..."
3,How to gain weight ?,How do I gain weight fast but still be healthy?,1,"[how, to, gain, weight]","[how, do, i, gain, weight, fast, but, still, b...","[how, to, gain, weight]","[how, do, i, gain, weight, fast, but, still, b...","[gain, weight]","[gain, weight, fast, still, healthy]"
4,Is porn bad for men?,Can I become a porn fan without getting addicted?,0,"[is, porn, bad, for, men]","[can, i, become, a, porn, fan, without, gettin...","[is, porn, bad, for, men]","[can, i, become, a, porn, fan, without, gettin...","[porn, bad, men]","[become, porn, fan, without, getting, addicted]"


In [8]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

MIN_FREQ = 2

counter = Counter()

for col in ["q1_tokens", "q2_tokens", "q1_unique", "q2_unique", "q1_keywords", "q2_keywords"]:
    for tokens in df[col]:
        counter.update(tokens)

vocab = [PAD_TOKEN, UNK_TOKEN]
vocab += [tok for tok, freq in counter.items() if freq >= MIN_FREQ]

stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

PAD_IDX = stoi[PAD_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]

print("Vocab size:", len(vocab))

Vocab size: 33664


In [9]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

MIN_FREQ = 2

counter = Counter()

for col in ["q1_tokens", "q2_tokens", "q1_unique", "q2_unique", "q1_keywords", "q2_keywords"]:
    for tokens in df[col]:
        counter.update(tokens)

vocab = [PAD_TOKEN, UNK_TOKEN]
vocab += [tok for tok, freq in counter.items() if freq >= MIN_FREQ]

stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

PAD_IDX = stoi[PAD_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]

print("Vocab size:", len(vocab))

Vocab size: 33664


In [10]:
MAX_LEN = 30  # common practical cap for Quora questions

def encode_tokens(tokens, stoi, max_len=MAX_LEN):
    ids = [stoi.get(tok, UNK_IDX) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))
    return ids

In [11]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["is_duplicate"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(train_df.shape, val_df.shape)

(40000, 9) (10000, 9)


In [12]:
class QuoraWECNNDataset(Dataset):
    def __init__(self, dataframe, stoi, max_len=30):
        self.df = dataframe
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        item = {
            "p": torch.tensor(encode_tokens(row["q1_tokens"], self.stoi, self.max_len), dtype=torch.long),
            "q": torch.tensor(encode_tokens(row["q2_tokens"], self.stoi, self.max_len), dtype=torch.long),

            "p_u": torch.tensor(encode_tokens(row["q1_unique"], self.stoi, self.max_len), dtype=torch.long),
            "q_u": torch.tensor(encode_tokens(row["q2_unique"], self.stoi, self.max_len), dtype=torch.long),

            "p_k": torch.tensor(encode_tokens(row["q1_keywords"], self.stoi, self.max_len), dtype=torch.long),
            "q_k": torch.tensor(encode_tokens(row["q2_keywords"], self.stoi, self.max_len), dtype=torch.long),

            "label": torch.tensor(int(row["is_duplicate"]), dtype=torch.long)
        }
        return item

In [13]:
BATCH_SIZE = 64

train_dataset = QuoraWECNNDataset(train_df, stoi, max_len=MAX_LEN)
val_dataset = QuoraWECNNDataset(val_df, stoi, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

p torch.Size([64, 30])
q torch.Size([64, 30])
p_u torch.Size([64, 30])
q_u torch.Size([64, 30])
p_k torch.Size([64, 30])
q_k torch.Size([64, 30])
label torch.Size([64])


In [14]:
from wecnn import WECNN

model = WECNN(
    vocab_size=len(vocab),
    embedding_dim=300,
    filters=(2, 3, 4, 6, 8),
    num_filters=200,
    pad_idx=PAD_IDX
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

WECNN(
  (embedding): Embedding(33664, 300, padding_idx=0)
  (keyword_embedding): Embedding(33664, 300, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(300, 200, kernel_size=(2,), stride=(1,))
    (1): Conv1d(300, 200, kernel_size=(3,), stride=(1,))
    (2): Conv1d(300, 200, kernel_size=(4,), stride=(1,))
    (3): Conv1d(300, 200, kernel_size=(6,), stride=(1,))
    (4): Conv1d(300, 200, kernel_size=(8,), stride=(1,))
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (bn): BatchNorm1d(300, eps=1e-05, momentum=0.7, affine=True, track_running_stats=True)
  (fc): Linear(in_features=45, out_features=2, bias=True)
)


In [15]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    for batch in tqdm(loader, desc="Training", leave=False):
        p = batch["p"].to(device)
        q = batch["q"].to(device)
        p_u = batch["p_u"].to(device)
        q_u = batch["q_u"].to(device)
        p_k = batch["p_k"].to(device)
        q_k = batch["q_k"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(p, q, p_u, q_u, p_k, q_k)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        probs = torch.softmax(logits, dim=1)
        pos_probs = probs[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_probs.extend(pos_probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    ll = log_loss(all_labels, all_probs)

    return avg_loss, acc, f1, auc, ll


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        p = batch["p"].to(device)
        q = batch["q"].to(device)
        p_u = batch["p_u"].to(device)
        q_u = batch["q_u"].to(device)
        p_k = batch["p_k"].to(device)
        q_k = batch["q_k"].to(device)
        labels = batch["label"].to(device)

        logits = model(p, q, p_u, q_u, p_k, q_k)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)

        probs = torch.softmax(logits, dim=1)
        pos_probs = probs[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_probs.extend(pos_probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    ll = log_loss(all_labels, all_probs)

    return avg_loss, acc, f1, auc, ll, all_labels, all_preds, all_probs

In [ ]:
EPOCHS = 5

best_val_auc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1, train_auc, train_logloss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, val_f1, val_auc, val_logloss, _, _, _ = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch}/{EPOCHS}")
    print(
        f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | Train AUC: {train_auc:.4f} | "
        f"Train LogLoss: {train_logloss:.4f}"
    )
    print(
        f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | "
        f"Val   F1: {val_f1:.4f} | Val   AUC: {val_auc:.4f} | "
        f"Val   LogLoss: {val_logloss:.4f}"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

Training:  11%|█         | 69/625 [00:04<00:26, 20.71it/s]

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

val_loss, val_acc, val_f1, val_auc, val_logloss, y_true, y_pred, y_prob = evaluate(
    model, val_loader, criterion, device
)

print("Best Validation Results")
print(f"Loss: {val_loss:.4f}")
print(f"Accuracy: {val_acc:.4f}")
print(f"F1-score: {val_f1:.4f}")
print(f"AUC: {val_auc:.4f}")
print(f"Log Loss: {val_logloss:.4f}")
print()
print(classification_report(y_true, y_pred, digits=4))

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Best Validation Results
Loss: 0.4932
Accuracy: 0.7835
F1-score: 0.6967
AUC: 0.8469

              precision    recall  f1-score   support

           0     0.8229    0.8406    0.8317      6362
           1     0.7104    0.6836    0.6967      3638

    accuracy                         0.7835     10000
   macro avg     0.7666    0.7621    0.7642     10000
weighted avg     0.7820    0.7835    0.7826     10000



In [ ]:
@torch.no_grad()
def predict_duplicate(model, q1, q2, stoi, max_len=30, device=device):
    model.eval()

    q1_tokens = tokenize(q1)
    q2_tokens = tokenize(q2)

    q1_unique = unique_in_order(q1_tokens)
    q2_unique = unique_in_order(q2_tokens)

    q1_keywords = extract_keywords(q1_tokens)
    q2_keywords = extract_keywords(q2_tokens)

    batch = {
        "p": torch.tensor([encode_tokens(q1_tokens, stoi, max_len)], dtype=torch.long).to(device),
        "q": torch.tensor([encode_tokens(q2_tokens, stoi, max_len)], dtype=torch.long).to(device),

        "p_u": torch.tensor([encode_tokens(q1_unique, stoi, max_len)], dtype=torch.long).to(device),
        "q_u": torch.tensor([encode_tokens(q2_unique, stoi, max_len)], dtype=torch.long).to(device),

        "p_k": torch.tensor([encode_tokens(q1_keywords, stoi, max_len)], dtype=torch.long).to(device),
        "q_k": torch.tensor([encode_tokens(q2_keywords, stoi, max_len)], dtype=torch.long).to(device),
    }

    logits = model(
        batch["p"], batch["q"],
        batch["p_u"], batch["q_u"],
        batch["p_k"], batch["q_k"]
    )

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

    return {
        "prediction": pred,
        "prob_not_duplicate": float(probs[0]),
        "prob_duplicate": float(probs[1]),
    }

In [ ]:
example1 = "How can I learn machine learning quickly?"
example2 = "What is the fastest way to study machine learning?"

result = predict_duplicate(model, example1, example2, stoi, max_len=MAX_LEN, device=device)
result

{'prediction': 1,
 'prob_not_duplicate': 0.21214927732944489,
 'prob_duplicate': 0.7878507971763611}